# Ride Share: SQL vs Redis + SQL trên Kaggle CPU
Chọn **Accelerator: None**, bật **Internet**, và dùng cùng một phiên cho cả hai chế độ. Notebook chạy native process, không yêu cầu Docker daemon.

Hai cách lấy mã nguồn: GitHub sau khi push implementation, hoặc attach Dataset chứa `kaggle-source.tar.gz` được tạo bằng `python3 scripts/package_kaggle.py`. Chọn archive để chạy chính xác thay đổi local chưa commit.

Notebook này được bàn giao chưa có kết quả thực thi trên tài khoản Kaggle. Chỉ coi kiểm chứng Kaggle hoàn tất khi **Save & Run All** thành công.


In [ ]:
from pathlib import Path
import json, os, subprocess, tarfile, sys
WORK = Path('/kaggle/working')
assert WORK.is_dir(), 'Run this notebook on Kaggle CPU.'
REPO_URL = 'https://github.com/tohuy2710/int3105-ride-share.git'
REVISION = 'main'  # After publishing: replace with the exact implementation commit SHA.
SOURCE_ARCHIVE = ''  # e.g. /kaggle/input/ride-share-source/kaggle-source.tar.gz
REPO = WORK / 'int3105-ride-share'
if SOURCE_ARCHIVE:
    assert not REPO.exists(), 'Use a fresh session before extracting a source archive.'
    with tarfile.open(SOURCE_ARCHIVE) as archive:
        archive.extractall(WORK, filter='data')
else:
    if not REPO.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVISION], check=True)
assert (REPO / 'scripts/kaggle_bootstrap.py').is_file(), 'Selected source does not contain the implementation yet.'
os.chdir(REPO)
print('Source:', REPO)


In [ ]:
TOOLS = WORK / 'ride-share-tools'
subprocess.run([sys.executable, 'scripts/kaggle_bootstrap.py', '--prefix', str(TOOLS)], check=True)
os.environ.update(json.loads((TOOLS / 'tools.env.json').read_text()))
os.environ['GOCACHE'] = str(WORK / 'go-build-cache')
os.environ['GOMODCACHE'] = str(WORK / 'go-module-cache')
os.environ['GOMAXPROCS'] = str(min(4, os.cpu_count() or 1))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'scripts/benchmark-requirements.txt'], check=True)
subprocess.run(['make', 'build', 'test', 'contract'], check=True)


## Kiểm tra nhanh trước khi đo
Bước này chạy PostgreSQL/Redis riêng, integration và fault tests với `-race`, rồi HTTP smoke cho cả 5 service. Các tiến trình được dọn trong `finally`; log vẫn được giữ lại.


In [ ]:
subprocess.run([sys.executable, 'scripts/native_check.py'], check=True)


## Đo hai chế độ trong cùng phiên
Mặc định: concurrency 2/10/50/100; 10 nhóm warm-up, 100 nhóm đo, 5 lần lặp cho hot/disjoint. Seed, login và kiểm tra DB nằm ngoài khoảng đo. `409` hợp lệ được báo riêng.


In [ ]:
OUTPUT = WORK / 'ride-share-results'
subprocess.run([sys.executable, 'scripts/benchmark_suite.py', '--output', str(OUTPUT)], check=True)
RUN = max(OUTPUT.glob('run-*'), key=lambda path: path.stat().st_mtime)
print('Artifacts:', RUN)
print((RUN / 'environment.json').read_text())
from IPython.display import display, Image
display(Image(filename=str(RUN / 'comparison.png')))


Kết quả gồm JSON/CSV thô, CPU/RAM theo thành phần, log, SHA256 binary và biểu đồ PNG/SVG. Load generator cùng dùng CPU với backend; không suy diễn kết quả này thành năng lực production. Bấm **Save & Run All** và giữ notebook version cùng thư mục output để làm bằng chứng.
